# ©️ Preprocesamiento y Modelado Predictivo

In [3]:
# Importaciones
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer

In [7]:
# Lectura del data frame y Limpieza de datos

df = pd.read_csv("../raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Funciones de Limpieza de Datos (Data Cleaning)

def strip_whitespace(dataframe):
    """Elimina espacios en blanco adicionales al inicio y final de los strings."""
    df_cleaned = dataframe.copy()
    for col in df_cleaned.select_dtypes(include=['object']).columns:
        df_cleaned[col] = df_cleaned[col].astype(str).str.strip()
    return df_cleaned

def fix_data_types(dataframe):
    """Corrige tipos de datos incorrectos, como convertir TotalCharges a numérico."""
    df_cleaned = dataframe.copy()
    if 'TotalCharges' in df_cleaned.columns:
        # Coerce reemplaza los espacios en blanco o textos inválidos con NaN
        df_cleaned['TotalCharges'] = pd.to_numeric(df_cleaned['TotalCharges'], errors='coerce')
    return df_cleaned

def handle_missing_values(dataframe):
    """Trata los valores nulos (NaN) imputando según el tipo de dato."""
    df_cleaned = dataframe.copy()
    for col in df_cleaned.columns:
        if df_cleaned[col].isnull().any():
            if df_cleaned[col].dtype in ['float64', 'int64']:
                # Imputar numéricos con la mediana
                median_value = df_cleaned[col].median()
                df_cleaned[col] = df_cleaned[col].fillna(median_value)
            else:
                # Imputar categóricos con la moda
                mode_value = df_cleaned[col].mode()[0]
                df_cleaned[col] = df_cleaned[col].fillna(mode_value)
    return df_cleaned

def drop_unnecessary_columns(dataframe, columns_to_drop=['customerID']):
    """Elimina columnas que no aportan valor predictivo como IDs."""
    df_cleaned = dataframe.copy()
    existing_cols = [col for col in columns_to_drop if col in df_cleaned.columns]
    return df_cleaned.drop(columns=existing_cols)

def clean_data(dataframe):
    """Aplica todo el flujo de limpieza de datos."""
    df_cleaned = strip_whitespace(dataframe)
    df_cleaned = fix_data_types(df_cleaned)
    df_cleaned = handle_missing_values(df_cleaned)
    df_cleaned = drop_unnecessary_columns(df_cleaned)
    return df_cleaned

# Ejecución de la limpieza sobre nuestro dataset actual
df_clean = clean_data(df)
print("¡Limpieza completada con éxito!")
print("Nuevos tipos de datos:")
print(df_clean.dtypes)

FileNotFoundError: [Errno 2] No such file or directory: '../raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'

# TABLA DE TIPOS DE VARIABLES 🧩🧩

In [ ]:
# Tabla de tipos de Variables (Modificada con mejores prácticas de Preprocesamiento)

def generar_tabla_variables(dataframe):
    variable_info = []
    
    # Clasificación y asignación de tratamientos óptimos basados en el preprocesamiento real
    for col in dataframe.columns:
        unique_vals = dataframe[col].dropna().unique()
        num_unique = len(unique_vals)
        dtype = str(dataframe[col].dtype)
        
        # Definición del tipo de variable y su respectivo tratamiento real aplicado
        if col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
            var_type = 'Numérica Continua/Discreta'
            tratamiento = 'Escalado (StandardScaler)'
        elif col == 'SeniorCitizen':
            var_type = 'Binaria (Numérica)'
            tratamiento = 'Dejala asi nomas'
        elif col == 'Churn':
            var_type = 'Variable Objetivo'
            tratamiento = 'No se escala ni se codifica como una variable mas'
        elif col in ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']:
            var_type = 'Binaria (Nominal)'
            tratamiento = 'Ordinal Encoding'
        elif col == 'Contract':
            var_type = 'Categórica Ordinal'
            tratamiento = 'Ordinal Encoding'
        else:
            var_type = 'Categórica Nominal'
            tratamiento = 'One Hot Encoding'
            
        variable_info.append({
            'Variable': col,
            'Tipo de Dato (Python)': dtype,
            'Tipo de Variable': var_type,
            'Valores Únicos': num_unique,
            'Ejemplos de Valores': list(unique_vals[:3]),
            'Tratamiento Real': tratamiento  
        })
        
    return pd.DataFrame(variable_info)

tabla_variables = generar_tabla_variables(df_clean)
tabla_variables

# Preprocesamiento de Datos para Modelado 🛠️

In [ ]:
# Definicion de que columnas tendra cada tratamiento
columnas_numericas = ['tenure', 'MonthlyCharges', 'TotalCharges']
columnas_binarias = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
columnas_ordinales = ['Contract']
columnas_nominales = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaymentMethod']

# Definicion del orden de contrato (0-2) de menor a mayor
ordenCorrecto_contrato = ['Month-to-month', 'One year', 'Two year']


# Preprocesador utilizando ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), columnas_numericas),
        ('bin_ord', OrdinalEncoder(), columnas_binarias),
        ('contract_ord', OrdinalEncoder(categories=[ordenCorrecto_contrato]), columnas_ordinales),
        ('nom_ohe', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), columnas_nominales)
    ],
    remainder='passthrough' # Para SeniorCitizen sin cambios es 0 y 1 XD
)

# Variables independientes (X) y variable objetivo (y)
X = df_clean.drop(columns=['Churn'])
y = LabelEncoder().fit_transform(df_clean['Churn'])

# Divicion entre Train y Test antes de transformar
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Ajuste del preprocesador con los datos de entrenamiento y transforme de ambos
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

# Convercion a DataFrame
encoded_cols = (
    columnas_numericas +
    columnas_binarias +
    columnas_ordinales +
    list(preprocessor.named_transformers_['nom_ohe'].get_feature_names_out(columnas_nominales)) +
    ['SeniorCitizen']
)

X_train_df = pd.DataFrame(X_train_prepared, columns=encoded_cols)
display(X_train_df.head())